In [5]:
using CSV, DataFrames, GLM, StatsPlots, Gurobi, JuMP
using LinearAlgebra, Random, DataFrames, CSV, Plots
using StatsBase, Statistics, Distributions
using JuMP, Gurobi
gurobi_env = Gurobi.Env()

Set parameter Username
Set parameter LicenseID to value 2749630
Academic license - for non-commercial use only - expires 2026-12-03


Gurobi.Env(Ptr{Nothing} @0x000000032c41da00, false, 0)

In [6]:
# Create a holistic regression model to predict with
#X = CSV.read("../clean_data/train_data_features.csv", DataFrame)
#y = CSV.read("../clean_data/test_data_features.csv", DataFrame)
;

In [30]:
using CSV
using DataFrames
using Statistics

# 1. Read CSV
df = CSV.read("../clean_data/train_data_features_24hr.csv", DataFrame)



# 2. Define target (y) and feature set (X)
# HB_NORTH is your response for holistic regression
y = df.HB_NORTH               # Vector{Float64}

# Drop the target and the timestamp from the features
feature_cols = Not([:HB_NORTH, :interval_start_local])
X_df = df[:, feature_cols]    # DataFrame of predictors

# 3. Handle missing values if there are any (simple example: fill with 0.0)
# You can replace 0.0 with mean/median/etc. if you prefer.
X_df = coalesce.(X_df, 0.0)

# 4. Ensure all features are Float64
X = Matrix{Float64}(X_df)     # n × p matrix of features
y_vec = Vector{Float64}(y)    # n-vector of targets

# 5. (Optional but common) Standardize features for regression
X_std = copy(X)
for j in 1:size(X_std, 2)
    μ = mean(X_std[:, j])
    σ = std(X_std[:, j])
    if σ > 0
        X_std[:, j] .= (X_std[:, j] .- μ) ./ σ
    end
end

# Now X_std (or X) and y_vec are ready for holistic regression
# Example (if you're using a HolisticRegression.jl-style API):
# using HolisticRegression
# model = HolisticRegression.fit(X_std, y_vec; λ=..., constraints=...)

In [51]:
using Random

function make_holdout(X, y; test_fraction = 0.2, seed = 123)
    Random.seed!(15059)
    n = size(X, 1)

    idx = shuffle(1:n)
    n_test = round(Int, test_fraction * n)

    test_idx = idx[1:n_test]
    train_idx = idx[(n_test+1):end]

    X_train = X[train_idx, :]
    y_train = y[train_idx]

    X_test  = X[test_idx, :]
    y_test  = y[test_idx]

    return X_train, y_train, X_test, y_test
end


X_train, y_train, X_hold, y_hold = make_holdout(X_std, y; test_fraction = 0.2);

In [52]:
size(X_train)

(13973, 45)

In [53]:
"""
Holistic regression MIP
"""       
XtX = X' * X
inv_XtX = inv(Symmetric(XtX))   # helps enforce symmetry numerically

diag_inv = diag(inv_XtX)
diag_inv = max.(diag_inv, 0.0)  # clip tiny negatives to 0

function compute_HC(X, ρ_max)
    n,p = size(X)
    c = zeros(p,p)
    for i=1:p-1,j=i+1:p
        c[i,j] = cor(X[:,i],X[:,j])
    end
    return [(i,j) for i=1:p for j=i+1:p if abs(c[i,j])>ρ_max]
end

function compute_sigma(X::Matrix{Float64}, y::Vector{Float64})
    n, p = size(X)
    if n <= p
        error("compute_sigma: need n > p (n = $n, p = $p)")
    end

    # OLS fit in a numerically stable way
    β̂ = X \ y                      # solves min ||Xβ - y||₂
    r  = y - X * β̂                 # residuals

    σ2 = dot(r, r) / (n - p)        # residual variance

    # clip potential tiny negative due to floating point
    σ2 = max(σ2, 0.0)

    return sqrt(σ2)
end
         
function holistic(X::Matrix{Float64}, y::Vector{Float64}, λ::Float64, mu::Float64,
                   ρ_max::Float64, t::Float64)
    M = 50    
    n,p = size(X)
    m = Model(() -> Gurobi.Optimizer(gurobi_env))
    set_optimizer_attribute(m, "OutputFlag", 0)
    #set_optimizer_attribute(m, "TimeLimit", 10000)
    
    @variable(m, β[1:p])
    @variable(m, z[1:p], Bin)
    @variable(m, b[1:p], Bin)    
    @variable(m, s[1:p])
    @variable(m, theta[1:n])

    # Objective
    @objective(m, Min, sum(theta) + λ * sum(s) + mu * sum(β.^2))
    
    @constraint(m, [i in 1:n], theta[i] >= (y[i] - dot(X[i, :], β))) 
    @constraint(m, [i in 1:n], theta[i] >= -(y[i] - dot(X[i, :], β)))   
    
    # 1-norm regularization
    @constraint(m, s .>= β)
    @constraint(m, s .>= -β)

    @constraint(m, [i=1:p], β[i] >= -M*z[i])
    @constraint(m, [i=1:p], β[i] <= M*z[i])

    # Pairwise correlation
    HC = compute_HC(X,ρ_max)
    for (i,j) in HC
        @constraint(m, z[i] + z[j] <= 1)
    end

    # Transformation 
    @constraint(m, [j in 1:4], z[j] + z[6+2(j-1)] + z[6+2(j-1)+1] <= 1)

    optimize!(m)
                    
    return value.(β), value.(z)
end

holistic (generic function with 1 method)

In [54]:
ρ_max = 0.9
t = 1.96
λ = 0.7
mu = 0.3

β2, z2 = holistic(X_std, y, λ, mu, ρ_max, t)

([0.0, 0.0, -1.2637745971016972, -2.0360499082138004, -5.828696128168963, 4.704938163924105, 0.0, 0.0, 3.7860486317578914, 0.0  …  0.0, -0.8585176404360179, -2.332740788585624, -0.05057786388966615, 0.0, 0.34991474655126475, 0.21435500544262032, -0.6485609734710787, -0.11119446090000716, 0.3372684161948821], [0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 1.0, 0.0  …  0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0])

In [55]:
# X_test y_test
using CSV
using DataFrames
using Statistics

# 1. Read CSV
df_t = CSV.read("../clean_data/test_data_features_24hr.csv", DataFrame)



# 2. Define target (y) and feature set (X)
# HB_NORTH is your response for holistic regression
y_t = df_t.HB_NORTH               # Vector{Float64}

# Drop the target and the timestamp from the features
feature_cols = Not([:HB_NORTH, :interval_start_local])
X_df_t = df_t[:, feature_cols]    # DataFrame of predictors

# 3. Handle missing values if there are any (simple example: fill with 0.0)
# You can replace 0.0 with mean/median/etc. if you prefer.
X_df_t = coalesce.(X_df_t, 0.0)

# 4. Ensure all features are Float64
X_t = Matrix{Float64}(X_df_t)     # n × p matrix of features
y_vec_t = Vector{Float64}(y_t)    # n-vector of targets

# 5. (Optional but common) Standardize features for regression
X_std_t = copy(X_t)
for j in 1:size(X_std_t, 2)
    μ = mean(X_std_t[:, j])
    σ = std(X_std_t[:, j])
    if σ > 0
        X_std_t[:, j] .= (X_std_t[:, j] .- μ) ./ σ
    end
end


In [56]:
function RMSE(y, pred) 
    return sqrt(sum((y-pred).^2)/length(y))
end;

function MAE(y, pred) 
    return mean(abs.((y-pred)))
end;

In [57]:
# Run test mse
test_RMSE = RMSE(y_t, X_std_t * β2)
test_MAE = MAE(y_t, X_std_t * β2)
println(test_RMSE)
println(test_MAE)

61.26402639766095
32.19258752635423


In [58]:
function baseline_day_before_test_rmse(y_test)
    H = 24
    
    # shift by 24 inside the test set itself
    y_pred = [fill(missing, H); y_test[1:end-H]]

    y_true = y_test[(H+1):end]
    y_pred_clean = y_pred[(H+1):end]

    return sqrt(mean((y_true .- y_pred_clean).^2))
end

rmse = baseline_day_before_test_rmse(y_t)
println("Test-set baseline MSE (day-before): ", rmse)

function baseline_day_before_test_mae(y_test)
    H = 24
    
    # shift by 24 inside the test set itself
    y_pred = [fill(missing, H); y_test[1:end-H]]

    y_true = y_test[(H+1):end]
    y_pred_clean = y_pred[(H+1):end]

    return mean(abs.(y_true .- y_pred_clean))
end

mae = baseline_day_before_test_mae(y_t)
println("Test-set baseline MAE (day-before): ", mae)

Test-set baseline MSE (day-before): 67.94895153263612
Test-set baseline MAE (day-before): 19.203570415040797


In [59]:
(rmse - test_RMSE)/rmse

0.09838157887932052

In [60]:
num_selected = count(==(1), z2)
println("Number of selected variables: $num_selected")

Number of selected variables: 24


In [61]:
selected = findall(z2 .== 1)

β2_selected = β2[selected]
abs_selected = abs.(β2_selected)

# Sort by absolute coefficient size
order = sortperm(abs_selected, rev=true)



selected_names = names(X_df)[selected]

println("\nStrongest Features:")
for idx in order
    println(selected_names[idx], ": ", abs_selected[idx])
end


Strongest Features:
time_hour_cos: 5.828696128168963
weather_temp_7d_mean: 4.704938163924105
solar_system_wide_lag24: 4.273312412635295
weather_CDD65_lag24: 3.7860486317578914
weather_slp_hPa_lag48: 2.998155909878227
wind_lz_west_lag24: 2.332740788585624
weather_temp_f_lag48: 2.195119635061457
load_system_lag48: 2.1535028409412784
time_hour_sin: 2.0360499082138004
solar_northwest_lag48: 1.8728001057589065
weather_dewpoint_C_lag48: 1.292108590375476
time_is_weekend: 1.2637745971016972
wind_lz_south_houston_lag24: 0.8585176404360179
wind_lz_north_lag48: 0.6485609734710787
weather_deltaT_dew_C_lag48: 0.6472648617623767
solar_centerwest_lag48: 0.6471168299563285
weather_HDD65_lag48: 0.44326934243165456
wind_lz_south_houston_lag48: 0.34991474655126475
price_lag48: 0.3372684161948821
weather_windspd_ms_lag48: 0.2935183694022992
wind_lz_west_lag48: 0.21435500544262032
price_lag24: 0.11119446090000716
weather_windspd_ms_lag24: 0.07640772455029994
wind_lz_north_lag24: 0.05057786388966615


In [62]:
using Pkg
Pkg.add("GLPK")

   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`


In [63]:
using JuMP
using GLPK  # or your favorite solver

"""
    quantile_regression(X, y; tau=0.5)

Solve quantile regression for given tau (median when tau=0.5).
X: Matrix (n, p)
y: Vector length n
"""
function quantile_regression(X, y; tau=0.5)
    n, p = size(X)
    X_design = hcat(ones(n), X)  # (n, p+1)
    p_full = p + 1

    model = Model(GLPK.Optimizer)

    @variable(model, β[1:p_full])
    @variable(model, r_pos[1:n] >= 0)  # max(residual, 0)
    @variable(model, r_neg[1:n] >= 0)  # max(-residual, 0)

    # residual: y - xᵢ'β = r_pos[i] - r_neg[i]
    @constraint(model, [i in 1:n],
        y[i] - sum(X_design[i, j] * β[j] for j in 1:p_full) == r_pos[i] - r_neg[i]
    )

    @objective(model, Min,
        tau * sum(r_pos[i] for i in 1:n) +
        (1 - tau) * sum(r_neg[i] for i in 1:n)
    )

    optimize!(model)

    return value.(β), objective_value(model)
end

# Example:
# β_opt, obj = quantile_regression(X, y; tau=0.5)  # median regression
# println("Intercept: ", β_opt[1])
# println("Slopes: ", β_opt[2:end])

quantile_regression

In [ ]:
n_test, p = size(X_std_t)

# Match what we did in training
X_test_design = hcat(ones(n_test), X_std_t)  # (n_test, p+1)

taus = 0.05:0.1:0.95


results = DataFrame(
    tau  = Float64[],
    mae  = Float64[],
    rmse = Float64[],
)

for τ in taus
    B_qr, obj_qr = quantile_regression(X_std, y, tau=τ)
    test_RMSE_qr = RMSE(y_t, X_test_design * B_qr)
    test_MAE_qr = MAE(y_t, X_test_design * B_qr)

    push!(results, (tau = τ, mae = test_MAE_qr, rmse = test_RMSE_qr))
end


In [67]:
function rank_column(values::AbstractVector{<:Real}; rev::Bool = false)
    # rev=false → lower value = better (rank 1)
    order = sortperm(values; rev=rev)
    ranks = similar(order)
    for (r, idx) in enumerate(order)
        ranks[idx] = r
    end
    return ranks
end

results.mae_rank  = rank_column(results.mae;  rev = false)  # smaller MAE is better
results.rmse_rank = rank_column(results.rmse; rev = false)  # smaller RMSE is better

results.combined_rank = results.mae_rank .+ results.rmse_rank

best_idx = argmin(results.combined_rank)
best_row = results[best_idx, :]

sorted_results = sort(results, :combined_rank)
first(sorted_results, 10)  # top 10

Row,tau,mae,rmse,mae_rank,rmse_rank,combined_rank
,Float64,Float64,Float64,Int64,Int64,Int64
1,0.5,18.7722,53.7216,1,1,2
2,0.6,18.7722,53.7216,2,2,4
3,0.7,18.7722,53.7216,3,3,6
4,0.8,18.7722,53.7216,4,4,8
5,0.9,18.7722,53.7216,5,5,10


In [28]:
# Run test mse
test_RMSE_qr = RMSE(y_t, X_test_design * B_qr)
test_MAE_qr = MAE(y_t, X_test_design * B_qr)
println(test_RMSE_qr)
println(test_MAE_qr)

53.35859623583825
18.375704115057395


In [29]:
test_prices = X_test_design * B_qr;

In [68]:
using CSV, DataFrames


df = DataFrame(value = test_prices)   # wrap vector in a one-column DataFrame
CSV.write("../price_data/lqad_q25.csv", df)

"../price_data/lqad_q25.csv"